## PBPK Modeling for VisiumHD
### pseudoVisium 55um

#### PBPK Modeling for ADC in VisiumHD 55um Grid Dataset

In [ ]:
import math
from pbpk_st import PBPKSimulation_ADC

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
import warnings
from pandas.errors import PerformanceWarning

warnings.simplefilter("ignore", PerformanceWarning)

In [ ]:
root_path = "../data/h5ad_55um_merged"
dataset_list = sorted([i for i in os.listdir(root_path) if i.endswith(".h5ad") and not ('B05' in i)])
dataset_list

In [ ]:
save_path = "../data/h5ad_pbpk_55um_subcluster_endo"

In [ ]:
if not os.path.exists(save_path):
    os.makedirs(save_path, exist_ok=True)

In [ ]:
cathepsin_genes = [
    "CTSB",  # Cathepsin B
    "CTSD",  # Cathepsin D
    "CTSL",  # Cathepsin L
    "CTSC",  # Cathepsin C
]

In [ ]:
endo_list = []
for dataset in dataset_list[1:]:
    print(f"Dataset Name: {dataset}")
    adata_grid = sc.read_h5ad(os.path.join(root_path, dataset))
    sc.pp.normalize_total(adata_grid, inplace=True)
    sc.pp.log1p(adata_grid)
    
    scalef = adata_grid.uns['spatial']['Visium_HD']['scalefactors']['microns_per_pixel']
    adata_grid.obsm['spatial'] = adata_grid.obsm['spatial'] * scalef
    adata_grid.obs["HER2_expression"] = adata_grid[:,'ERBB2'].X.toarray()
    adata_grid.obs["endothelial_density"] = adata_grid.obs["Endothelial_filtered"]
    endo_max = adata_grid.obs["endothelial_density"].max()
    print(endo_max)
    endo_list.append(endo_max)
    print("-"*30)

In [ ]:
endo_scalef = np.median(endo_list)

In [ ]:
for dataset in dataset_list:
    print(f"Dataset Name: {dataset}")
    dataset_name = dataset.split('_')[2]
    adata_grid = sc.read_h5ad(os.path.join(root_path, dataset))
    sc.pp.normalize_total(adata_grid, inplace=True)
    sc.pp.log1p(adata_grid)
    
    scalef = adata_grid.uns['spatial']['Visium_HD']['scalefactors']['microns_per_pixel']
    adata_grid.obsm['spatial'] = adata_grid.obsm['spatial'] * scalef
    adata_grid.obs["HER2_expression"] = adata_grid[:,'ERBB2'].X.toarray()
    adata_grid.obs["endothelial_density"] = adata_grid.obs["Endothelial_filtered"] / endo_scalef
    # adata_grid.obs["endothelial_density"] = adata_grid.obs["Endothelial_filtered"] / np.max(adata_grid.obs["Endothelial_filtered"])
    # Score genes related to linker enzyme activity
    sc.tl.score_genes(adata_grid, gene_list=cathepsin_genes, score_name="linker_enzyme", use_raw=False)
    adata_grid.obs["linker_enzyme"] = np.clip(adata_grid.obs["linker_enzyme"], a_min=0, a_max=None)
    print(f'Max value: {adata_grid.obs["linker_enzyme"].max()}')

    for subcluster in np.setdiff1d(pd.unique(adata_grid.obs['subcluster']), -1):
        adata_grid_sub = adata_grid[adata_grid.obs['subcluster']==subcluster].copy()
        core_name = adata_grid_sub.obs['core_name'].values[0]
        print(f"Core-Subcluster Name: {core_name}-{subcluster}")
        try:
            # Run PBPK ADC simulation
            sim_adc = PBPKSimulation_ADC(
                    adata=adata_grid_sub,
                    target_column="HER2_expression",
                    endothelial_column="endothelial_density",
                    linker_enzyme_column="linker_enzyme",
                    DAR=8.0,  # Drug-to-antibody ratio
                    scaling_factor=1.0,
                    kon=0.71,  # μM^-1 s^-1
                    Kd=5e-4,  # μM
                    k_internalize=math.log(2) / (3600 * 3),  # s^-1 s^-1 1e-4 about 2hr, 1e-3 --> 11.6min 
                    k_cleavage=math.log(2) / (3600 * 5),  # s^-1·μM^-1 cleavage rate
                    k_clear=math.log(2) / (3600 * 24 * 7),  # Antibody clearance rate
                    k_clearance_payload=math.log(2) / (3600 * 1.37),  # Payload clearance rate
                    initial_endothelial_concentration=0.8,  # Initial antibody concentration (μM per endothelial density)
                    r_antibody=5e-9,  # Antibody radius (m)
                    k_neighbors=30  # Number of neighbors for diffusion
            )
        
            result_adata_adc = sim_adc.run(
                t_span=(0, 72000),  # Simulate for 20 hours
                steps=100,  # 100 time points,
                output_pd=False
            )
            
            result_adata_adc.obsm['spatial'] = result_adata_adc.obsm['spatial'] / scalef
            result_adata_adc.write_h5ad(os.path.join(save_path, f'PBPK_ADC_{dataset_name}_{core_name}_{subcluster}_HER2_Endo.h5ad'))
        except Exception as e:
            print(f"An actual error occurred: {e}")  
    print("-"*30)